In [7]:
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.3.230  Python-3.10.6 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
Setup complete  (32 CPUs, 31.7 GB RAM, 180.0/234.8 GB disk)


In [10]:
import torch
# 1. Hardware Verification
# This ensures PyTorch can actually "see" your CUDA installation.
# If this prints False, your Jupyter kernel lacks the CUDA toolkit bindings.
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# 2. Load the base model architecture
model = YOLO('yolov8n.pt')

# 3. GPU-Optimized Training Loop
results = model.train(
    data='dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=32,        # Increased to saturate the 12GB VRAM
    device=0,        # Explicitly targets the first CUDA device (your RTX 4080)
    workers=8,       # Background CPU threads to keep the GPU fed with data
    name='ambulance_run_gpu'
)

CUDA Available: True
Active GPU: NVIDIA GeForce RTX 4070 Laptop GPU
New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.230  Python-3.10.6 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=

In [12]:
# This automatically loads the 'best.pt' weights generated from your training run
# and calculates the Mean Average Precision (mAP).
metrics = model.val()

print(f"mAP50: {metrics.box.map50:.3f}")

Ultralytics 8.3.230  Python-3.10.6 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 53.716.7 MB/s, size: 25.1 KB)
val: Scanning C:\Users\shaik\Desktop\training\dataset\labels\val.cache... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 0/3  5.0s


RuntimeError: DataLoader worker (pid(s) 38336, 16272, 38540, 39016, 30836, 37048, 10448, 37568) exited unexpectedly

In [ ]:
import os
import random

# Grab the first image from your validation set
val_dir = 'dataset/images/val/'
val_images = [f for f in os.listdir(val_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
random_image = random.choice(val_images)
sample_image = os.path.join(val_dir, os.listdir(val_dir)[0])

# Run inference. save=True outputs a new image with the bounding box drawn on it
# inside the runs/detect/predict directory.
inference_results = model.predict(source=sample_image, save=True, conf=0.5)


image 1/1 c:\Users\shaik\Desktop\training\dataset\images\val\b86aed53-20.jpg: 352x640 1 Ambulance, 43.7ms
Speed: 3.3ms preprocess, 43.7ms inference, 11.6ms postprocess per image at shape (1, 3, 352, 640)
Results saved to C:\Users\shaik\Desktop\training\runs\detect\predict
